# 03 · Ablation studies

Each ablation is one config in `configs/ablations/`, trained with the same shorter schedule of 100 epochs (about 1 hour on a T4). Run them **one at a time**:
pick `ABLATION`, then `Runtime → Run all`. If Colab disconnects, run it again and it resumes.

| group | runs |
|---|---|
| attention | `a01_no_attention`, `a02_channel_only`, `a02b_eca`, `a03_spatial_only`, `a04_full`, `a05_no_gates` |
| loss | `l1_charbonnier`, `l2_ssim`, `l3_perceptual` (+ `a04_full`) |
| width | `w08`, `w32` (+ `a04_full`) |

`a04_full` is the reference row of all three tables, so **train it first**. The last cell shows which runs are done and builds the tables.

## Setup
Runs in about 1–2 minutes. It mounts Google Drive, gets the latest code, installs the requirements, and restores the datasets from the zip cache on your Drive. Run `00_setup_and_data.ipynb` once beforehand.

In [ ]:
REPO_URL = "https://github.com/Asad3404/LUMA.git"
BRANCH = "claude/attention-unet-low-light-ay0gpv"   # change to "main" once merged
DRIVE_ROOT = "/content/drive/MyDrive/lowlight-attn-unet"
DATA_ROOT = "/content/data"
REPO_DIR = "/content/LUMA"

from google.colab import drive
drive.mount("/content/drive")
import os
if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="code version: %h  %s"
!pip install -q -r requirements.txt 
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!python data/download.py --datasets lol_v1 --data_root {DATA_ROOT} --cache_dir {DRIVE_ROOT}/datasets --results_dir {DRIVE_ROOT}/results/data

In [ ]:
ABLATION = "a04_full"   # <- change this each session
MAX_HOURS = 2.6
RESULTS = f"{DRIVE_ROOT}/results"
RUNS = f"{DRIVE_ROOT}/runs"

## Train (resumes automatically)

In [ ]:
!python src/train.py --config configs/ablations/{ABLATION}.yaml --max_hours {MAX_HOURS}

## Evaluate on LOL-v1 and benchmark
Only runs once training has finished.

In [ ]:
import json, os
summ = f"{RUNS}/{ABLATION}/train_summary.json"
if os.path.exists(summ) and json.load(open(summ))["status"] == "finished":
    !python src/evaluate.py --checkpoint {RUNS}/{ABLATION}/best.pth --method {ABLATION} --datasets lol_v1 --out {RESULTS}
    !python src/benchmark.py --checkpoint {RUNS}/{ABLATION}/best.pth --method {ABLATION} --out {RESULTS} --runs 100 --sizes 600x400
else:
    print("Training not finished yet: run this notebook again to resume.")

## Progress overview and tables

In [ ]:
import glob, yaml
idx = yaml.safe_load(open("configs/ablations/index.yaml"))
runs = sorted({r["run"] for g in idx.values() for r in g["rows"]})
for r in runs:
    s = f"{RUNS}/{r}/train_summary.json"
    st = json.load(open(s))["status"] if os.path.exists(s) else "not started"
    ev = "evaluated" if os.path.exists(f"{RESULTS}/metrics/{r}/lol_v1.json") else ""
    print(f"{r:<20} {st:<30} {ev}")
!python src/collect_results.py --results {RESULTS} --no_readme | sed -n '/Ablation/,$p'

When every run is done, run the last section of notebook 02 (sync + push) to copy the ablation tables into the repo.